In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


output_notebook()
hv.extension('bokeh')

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

Loading BokehJS ...

In [2]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 
base_path = Path.cwd().parent / 'data' / 'csst_trials_pkls'
# filepath = base_path / f'all_{monkey}_CSST_trials_df.pkl'
# filepath = base_path / f'all_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'
# filepath = base_path / f'no_filters_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'
filepath = base_path / f'alt_saccade_params_{monkey}_CSST_trials_df_including_all_200_neuron_fields.pkl'

df = pd.read_pickle(filepath)

print(df.info())
# df.iloc[:2]
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110358 entries, 0 to 110357
Data columns (total 28 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   blinks                  21339 non-null   object 
 1   dir                     110358 non-null  int64  
 2   direction               110358 non-null  object 
 3   filename                110358 non-null  object 
 4   first_relevant_saccade  98324 non-null   object 
 5   flags                   110358 non-null  int64  
 6   go_cue                  110358 non-null  int64  
 7   hPos                    110358 non-null  object 
 8   hVel                    110358 non-null  object 
 9   neural_data             110358 non-null  object 
 10  saccades                110206 non-null  object 
 11  screen_rotation         110358 non-null  float64
 12  segs_durations          110358 non-null  object 
 13  segs_times              110358 non-null  object 
 14  set                 

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel,reaction_time
0,None,0,R,fi210713a.0529,NaN,11278,921,"[28.55, 28.55, 28.55, 28.55, 28.55, 28.55, 28....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1005.0,False,1605,STOP_R_SSD2,0529,fi210713a,STOP,"[-11.575, -11.575, -11.575, -11.6, -11.6, -11....","[-4.686380092992484, -4.686380092992484, -4.04...",NaN
1,None,0,R,fi210713a.0169,"[1157, 1190]",9222,921,"[-13.225, -13.225, -13.25, -13.175, -13.175, -...","[-3.951261647032878, -3.951261647032878, -5.51...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1125.0,True,1726,STOP_R_SSD4,0169,fi210713a,STOP,"[-0.5, -0.5, -0.525, -0.525, -0.525, -0.5, -0....","[-1.3783470861742597, -1.3783470861742597, -0....",236.0
2,"[338, 444, 447, 449]",180,L,fi210713a.0457,NaN,11278,1093,"[-11.775, -11.775, -11.775, -11.775, -11.775, ...","[-1.286457280429309, -1.286457280429309, -0.91...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1117.0,False,1717,STOP_L_SSD1,0457,fi210713a,STOP,"[-0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0....","[-3.6755922297980264, -3.6755922297980264, -1....",NaN
3,None,0,R,fi210713a.0693,"[1254, 1286]",8206,1043,"[-0.475, -0.475, -0.55, -0.55, -0.55, -0.55, -...","[0.7351184459596053, 0.7351184459596053, -2.75...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,NaN,False,2194,GO_R,0693,fi210713a,GO,"[-0.75, -0.75, -0.775, -0.775, -0.775, -0.775,...","[2.572914560858618, 2.572914560858618, -0.4594...",211.0
4,None,180,L,fi210713a.0475,"[1207, 1241]",8206,1018,"[2.325, 2.325, 2.325, 2.3, 2.3, 2.3, 2.3, 2.3,...","[2.2972451436237664, 2.2972451436237664, 2.113...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,NaN,False,2169,GO_L,0475,fi210713a,GO,"[-0.575, -0.575, -0.475, -0.55, -0.55, -0.55, ...","[1.5621266976641612, 1.5621266976641612, 3.491...",189.0


In [3]:
df['filename'].apply(lambda x: x.split('.')[0][-1]).unique()

array(['a'], dtype=object)

In [4]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['trial_session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

df.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}') 

Total unique neurons across all sessions: 17600


In [5]:
# Drop unnecessary columns
cols_to_drop = [
    'vPos', 'hPos', 'vVel', 'hVel', 'speed',
    'set', 'direction'
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"DataFrame shape after dropping columns: {df.shape}")
df.head()

DataFrame shape after dropping columns: (110358, 21)


,blinks,dir,filename,first_relevant_saccade,flags,go_cue,neural_data,saccades,screen_rotation,segs_durations,...,ssd_len,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,reaction_time
0,None,0,fi210713a.0529,NaN,11278,921,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[248, 296], [569, 586]]",0.0,"[500, 421, 84, 600]",...,84,2.0,1005.0,False,1605,STOP_R_SSD2,0529,fi210713a,STOP,NaN
1,None,0,fi210713a.0169,"[1157, 1190]",9222,921,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[36, 63], [364, 397], [1157, 1190]]",0.0,"[500, 421, 204, 600]",...,204,4.0,1125.0,True,1726,STOP_R_SSD4,0169,fi210713a,STOP,236.0
2,"[338, 444, 447, 449]",180,fi210713a.0457,NaN,11278,1093,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[133, 165], [323, 352], [444, 526], [536, 547]]",0.0,"[500, 593, 24, 600]",...,24,1.0,1117.0,False,1717,STOP_L_SSD1,0457,fi210713a,STOP,NaN
3,None,0,fi210713a.0693,"[1254, 1286]",8206,1043,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[283, 305], [439, 454], [1254, 1286]]",0.0,"[500, 543, 550, 600]",...,550,NaN,NaN,False,2194,GO_R,0693,fi210713a,GO,211.0
4,None,180,fi210713a.0475,"[1207, 1241]",8206,1018,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[188, 209], [1207, 1241]]",0.0,"[500, 518, 550, 600]",...,550,NaN,NaN,False,2169,GO_L,0475,fi210713a,GO,189.0


In [6]:
# reorder columns
new_order = [
    'filename', 'trial_name', 'reaction_time', 
    'go_cue', 'stop_cue', 'trial_failed', 
    'first_relevant_saccade', 'segs_durations', 'segs_times',
    'trial_length', 'ssd_len', 'ssd_number',
    'screen_rotation', 'neural_data', 'saccades', 
    'blinks', 'dir', 'flags',
    'type', 'trial_session', 'trial_number',
]

df = df[new_order]
df.head()

,filename,trial_name,reaction_time,go_cue,stop_cue,trial_failed,first_relevant_saccade,segs_durations,segs_times,trial_length,...,ssd_number,screen_rotation,neural_data,saccades,blinks,dir,flags,type,trial_session,trial_number
0,fi210713a.0529,STOP_R_SSD2,NaN,921,1005.0,False,NaN,"[500, 421, 84, 600]","[0, 500, 921, 1005, 1605]",1605,...,2.0,0.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[248, 296], [569, 586]]",None,0,11278,STOP,fi210713a,0529
1,fi210713a.0169,STOP_R_SSD4,236.0,921,1125.0,True,"[1157, 1190]","[500, 421, 204, 600]","[0, 500, 921, 1125, 1725]",1726,...,4.0,0.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[36, 63], [364, 397], [1157, 1190]]",None,0,9222,STOP,fi210713a,0169
2,fi210713a.0457,STOP_L_SSD1,NaN,1093,1117.0,False,NaN,"[500, 593, 24, 600]","[0, 500, 1093, 1117, 1717]",1717,...,1.0,0.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[133, 165], [323, 352], [444, 526], [536, 547]]","[338, 444, 447, 449]",180,11278,STOP,fi210713a,0457
3,fi210713a.0693,GO_R,211.0,1043,NaN,False,"[1254, 1286]","[500, 543, 550, 600]","[0, 500, 1043, 1593, 2193]",2194,...,NaN,0.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[283, 305], [439, 454], [1254, 1286]]",None,0,8206,GO,fi210713a,0693
4,fi210713a.0475,GO_L,189.0,1018,NaN,False,"[1207, 1241]","[500, 518, 550, 600]","[0, 500, 1018, 1568, 2168]",2169,...,NaN,0.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...","[[188, 209], [1207, 1241]]",None,180,8206,GO,fi210713a,0475


In [7]:
# load monkey's cell db from xlsx file
cell_db_path = Path.cwd().parent / 'data' / f'database_sst'
cell_db = pd.read_excel(cell_db_path / f'SST_{monkey}_cells_db.xlsx')
print(cell_db.info())
cell_db.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5309 entries, 0 to 5308
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cell_ID              5309 non-null   int64  
 1   session              5309 non-null   object 
 2   cell_type            5309 non-null   object 
 3   electrode            5309 non-null   int64  
 4   template             5309 non-null   int64  
 5   maestro_ID           5309 non-null   int64  
 6   phy_id               0 non-null      float64
 7   phy_channel          0 non-null      float64
 8   file_begin           5309 non-null   int64  
 9   file_end             5309 non-null   int64  
 10  fb_after_stablility  5309 non-null   object 
 11  fe_after_stability   5309 non-null   object 
 12  plexon_session       5309 non-null   object 
 13  grade                5309 non-null   int64  
 14  X                    5309 non-null   int64  
 15  Y                    5309 non-null   i

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
0,9001,fi210628,ctx,1,1,1,NaN,NaN,1,225,...,0,0,4420.0,2,depth micro m is from cortex surface,fi210628a-01.pl2,NaN,1,broken cell two peaks,1
1,9002,fi210629,msn,1,1,1,NaN,NaN,84,163,...,0,-1,10200.0,2,NaN,fi210629a-01.pl2,NaN,1,NaN,1
2,9003,fi210701,msn,1,1,1,NaN,NaN,16,348,...,0,-1,7030.0,2,2 cells multi unit,fi210701a-02.pl2,NaN,1,NaN,1
3,9004,fi210701,tan,1,1,1,NaN,NaN,350,510,...,0,-1,7110.0,2,NaN,fi210701b-02.pl2,NaN,1,NaN,1
4,9005,fi210701,msn,1,2,2,NaN,NaN,16,348,...,0,-1,7030.0,2,NaN,fi210701a-02.pl2,NaN,1,NaN,1


In [8]:
cell_db[
    (cell_db['cell_type'] == 'msn') & 
    (cell_db['grade'] <= 8)
].shape

(1307, 24)

In [9]:
df.iloc[0].neural_data.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199])

In [10]:
if monkey == 'fiona':
    cell_db.at[429, 'fe_after_stability'] = 1017

In [11]:
cell_db.columns

Index(['cell_ID', 'session', 'cell_type', 'electrode', 'template',
       'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end',
       'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade',
       'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file',
       'tmp', 'sorted', 'problem', 'synced_stability'],
      dtype='object')

In [12]:
cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].apply(
    lambda row: [
        np.fromstring(
            row[key][1:-1], sep=' ', dtype=np.int16
        )
        for key in ['fb_after_stablility', 'fe_after_stability']
    ],
    axis=1, result_type='expand'
)

# np.fromstring(tmp[1:-1], sep=' ', dtype=np.int16)

,0,1
729,"[651, 1163]","[1066, 1279]"
741,"[741, 854]","[814, 949]"
750,"[332, 594]","[519, 644]"
777,"[693, 1235, 1450]","[1123, 1418, 1584]"
795,"[1, 399]","[314, 637]"
...,...,...
3627,"[1, 253]","[116, 630]"
3644,"[1, 254]","[113, 641]"
4149,"[981, 1748]","[1724, 2270]"
4644,"[1054, 1592, 1731, 1965]","[1556, 1700, 1924, 2382]"


In [13]:
def get_row_stable_trials_total(row):
    if isinstance(row['fe_after_stability'], str):
        fb_stable = np.fromstring(
            row['fb_after_stablility'][1:-1], sep=' ', dtype=np.int16
        )
        fe_stable = np.fromstring(
            row['fe_after_stability'][1:-1], sep=' ', dtype=np.int16
        )
        return fe_stable.sum() - fb_stable.sum()
    elif isinstance(row['fe_after_stability'], int):
        return row['fe_after_stability'] - row['fb_after_stablility']
    else:
        raise ValueError("Unexpected data type in 'fe_after_stability' column")
    
row = cell_db.iloc[0]
row = cell_db[
    cell_db['fe_after_stability'].apply(
        lambda row: isinstance(row, str)
)].iloc[0]
get_row_stable_trials_total(row)
# cell_db[(cell_db.apply(get_row_stable_trials_total, axis=1) < 0)]
stable_trials = cell_db[cell_db['cell_type'].isin(['msn', 'pu msn'])].apply(get_row_stable_trials_total, axis=1)
stable_trials.sum()

np.int64(2807528)

In [14]:
# Function to extract session and trial number from filename
def parse_filename(filename):
    """
    Parse filename like 'fi210824a.0614' into components
    Returns: (session, plexon_session, trial_number)
    """
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

# Test the function
test_filename = df.iloc[0]['filename']
print(f"Test filename: {test_filename}")
session, plexon_session, trial_num = parse_filename(test_filename)
print(f"Parsed: session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

# Check a few more examples
print("\nTesting with more filenames:")
for i in range(5):
    fname = df.iloc[i]['filename']
    session, plexon_session, trial_num = parse_filename(fname)
    print(f"{fname} -> session='{session}', plexon_session='{plexon_session}', trial_num={trial_num}")

Test filename: fi210713a.0529
Parsed: session='fi210713', plexon_session='a', trial_num=529

Testing with more filenames:
fi210713a.0529 -> session='fi210713', plexon_session='a', trial_num=529
fi210713a.0169 -> session='fi210713', plexon_session='a', trial_num=169
fi210713a.0457 -> session='fi210713', plexon_session='a', trial_num=457
fi210713a.0693 -> session='fi210713', plexon_session='a', trial_num=693
fi210713a.0475 -> session='fi210713', plexon_session='a', trial_num=475


In [15]:
# Check the column names for stability columns
print("Cell DB columns:")
print(cell_db.columns.tolist())

print(f"\nSample stability columns:")
stability_cols = ['fb_after_stablility', 'fe_after_stability']
for col in stability_cols:
    if col in cell_db.columns:
        print(f"{col}: {cell_db[col].head().tolist()}")
        print(f"  Data types: {cell_db[col].apply(type).value_counts()}")
    else:
        print(f"❌ Column '{col}' not found!")

# Check neural data keys format
print(f"\nSample neural data keys:")
sample_neural_data = df.iloc[0]['neural_data']
print(f"Type: {type(sample_neural_data)}")
if isinstance(sample_neural_data, dict):
    print(f"Keys (first 10): {list(sample_neural_data.keys())[:10]}")
    print(f"Key types: {[type(k) for k in list(sample_neural_data.keys())[:5]]}")
else:
    print(f"Not a dict: {sample_neural_data}")

Cell DB columns:
['cell_ID', 'session', 'cell_type', 'electrode', 'template', 'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end', 'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade', 'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file', 'tmp', 'sorted', 'problem', 'synced_stability']

Sample stability columns:
fb_after_stablility: [1, 84, 16, 350, 16]
  Data types: fb_after_stablility
<class 'int'>    5178
<class 'str'>     131
Name: count, dtype: int64
fe_after_stability: [225, 163, 348, 510, 348]
  Data types: fe_after_stability
<class 'int'>    5178
<class 'str'>     131
Name: count, dtype: int64

Sample neural data keys:
Type: <class 'dict'>
Keys (first 10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Key types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]


In [16]:
# Helper functions for parallel processing
def is_trial_in_stability_range(fb_stability, fe_stability, trial_num):
    """Check if trial is within ANY stability range, handling multiple ranges properly"""
    # Wrap arguments in lists and create numpy arrays, then flatten
    try:
        fb_array = np.array([int(fb_stability)])
        fe_array = np.array([int(fe_stability)])
    except ValueError as e:
        if str(e).startswith("invalid literal for int() with base 10: "):
            fb_array = np.fromstring(fb_stability[1:-1], sep=' ', dtype=np.int16)
            fe_array = np.fromstring(fe_stability[1:-1], sep=' ', dtype=np.int16)
        else:
            raise e
    
    # Get minimum length to ensure we don't go out of bounds
    min_len = min(len(fb_array), len(fe_array))
    
    # Check each stability range pair
    for i in range(min_len):
        fb_start = fb_array[i]
        fe_end = fe_array[i]
        
        # Skip invalid values
        if pd.isna(fb_start) or pd.isna(fe_end):
            continue
            
        try:
            # Check if trial falls within this range
            if int(fb_start) <= trial_num <= int(fe_end):
                return True  # Trial is within this stability range
        except (ValueError, TypeError):
            continue  # Skip this range if conversion fails
    
    return False  # Trial is not within any stability range

def parse_filename(filename):
    """Parse filename like 'fi210824a.0614' into components"""
    parts = filename.split('.')
    if len(parts) != 2:
        return None, None, None
    
    prefix = parts[0]  # 'fi210824a'
    trial_num = parts[1]  # '0614'
    
    if len(prefix) < 9:  # minimum: 'fi' + 6 digits + 'a'
        return None, None, None
    
    session = prefix[:-1]  # 'fi210824' (remove plexon session letter)
    plexon_session = prefix[-1]  # 'a'
    
    return session, plexon_session, int(trial_num)

def process_trial_chunk(args):
    """Process a chunk of trials - this function will run in parallel"""
    trial_chunk, cells_df_dict = args
    
    # Convert cells_df_dict back to DataFrame
    cells_df = pd.DataFrame(cells_df_dict)
    
    unified_data = []
    
    for _, trial_row in trial_chunk.iterrows():
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                'grade': cell_row['grade'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    return unified_data

# PARALLEL VERSION - Create the CORRECTED unified DataFrame with ProcessPoolExecutor
def create_unified_dataframe_parallel(trials_df, cells_df, chunk_size=100, max_workers=None):
    """
    Create a unified DataFrame with one row per cell-trial combination using parallel processing.
    
    Parameters:
    -----------
    trials_df : pd.DataFrame
        DataFrame with trial data
    cells_df : pd.DataFrame  
        DataFrame with cell information
    chunk_size : int
        Number of trials to process in each chunk
    max_workers : int
        Number of parallel workers (None = use CPU count)
    
    Returns:
    --------
    pd.DataFrame : Unified DataFrame with cell-trial combinations
    """
    print("Creating unified DataFrame with corrections (PARALLEL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    # Convert cells_df to dict for pickling (required for ProcessPoolExecutor)
    cells_df_dict = cells_df.to_dict('records')
    
    # Split trials into chunks
    trial_chunks = []
    for i in range(0, len(trials_df), chunk_size):
        chunk = trials_df.iloc[i:i+chunk_size]
        trial_chunks.append((chunk, cells_df_dict))
    
    print(f"Split into {len(trial_chunks)} chunks of ~{chunk_size} trials each")
    
    # Determine number of workers
    if max_workers is None:
        max_workers = min(mp.cpu_count(), len(trial_chunks))
    
    print(f"Using {max_workers} parallel workers")
    
    # Process chunks in parallel
    all_unified_data = []
    all_unstable_trials = []    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_trial_chunk, args): i 
                  for i, args in enumerate(trial_chunks)}
        
        # Collect results with progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing chunks"):
            chunk_results = future.result()
            all_unified_data.extend(chunk_results)
    
    # Create final DataFrame
    unified_df = pd.DataFrame(all_unified_data)

    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Original sequential version (for comparison or fallback)
def create_unified_dataframe_corrected(trials_df, cells_df):
    """Sequential version - kept for fallback or comparison"""
    unified_data = []
    
    print("Creating unified DataFrame with corrections (SEQUENTIAL VERSION)...")
    print(f"Processing {len(trials_df)} trials and {len(cells_df)} cells...")
    
    for trial_idx, trial_row in tqdm(trials_df.iterrows(), total=len(trials_df), desc="Processing trials"):
        # Parse filename to get session info
        filename = trial_row['filename']
        session, plexon_session, trial_num = parse_filename(filename)
        
        if session is None:
            # raise ValueError(f"Could not parse filename: {filename}")
            continue
        
        # Get neural data for this trial
        # neural_data = trial_row.get('neural_data', {})
        neural_data = trial_row['neural_data']
        if not isinstance(neural_data, dict):
            # neural_data = {}
            raise ValueError(f"Unexpected neural_data format in trial '{filename}'")
        
        # Find cells that match session and plexon session
        session_cells = cells_df[
            (cells_df['session'] == session) #& 
            # (cells_df['plexon_session'] == plexon_session) !!!!!!!!!! This could be the issue
        ]
        
        # Filter by stability range
        stable_cells = []
        for _, cell_row in session_cells.iterrows():
            if is_trial_in_stability_range(
                cell_row['fb_after_stablility'], 
                cell_row['fe_after_stability'], 
                trial_num
            ):
                stable_cells.append(cell_row)
            else:
                error = f"Trial {trial_num} not in stability range for cell {cell_row['cell_ID']} ins session {session}"
                raise ValueError(error)
        
        # Create one row per cell for this trial
        for cell_row in stable_cells:
            maestro_id_matlab = cell_row['maestro_ID']  # MATLAB 1-based ID
            
            # Fix MATLAB/Python indexing: MATLAB uses 1-based, Python uses 0-based
            # Neural data keys are 0-based (Python), maestro_ID in xlsx is 1-based (MATLAB)
            maestro_id_python = maestro_id_matlab - 1  # Convert to Python 0-based
            cell_neural_data = neural_data.get(maestro_id_python, [])
            
            # Create unified row
            unified_row = {
                # Cell information
                'cell_ID': cell_row['cell_ID'],
                'cell_type': cell_row['cell_type'], 
                'maestro_ID': maestro_id_matlab,  # Keep original MATLAB ID for reference
                'problem': cell_row['problem'],
                
                # Core trial information
                'filename': trial_row['filename'],
                'trial_name': trial_row['trial_name'],
                'reaction_time': trial_row['reaction_time'],
                'go_cue': trial_row['go_cue'],
                'stop_cue': trial_row['stop_cue'], 
                'trial_failed': trial_row['trial_failed'],
                'ssd_len': trial_row['ssd_len'],
                'ssd_number': trial_row['ssd_number'],
                'type': trial_row['type'],
                
                # Additional trial information
                'first_relevant_saccade': trial_row['first_relevant_saccade'],
                'segs_durations': trial_row['segs_durations'],
                'segs_times': trial_row['segs_times'],
                'trial_length': trial_row['trial_length'],
                'screen_rotation': trial_row['screen_rotation'],
                'saccades': trial_row['saccades'],
                'blinks': trial_row['blinks'],
                'dir': trial_row['dir'],
                
                # Neural data for this specific cell
                'neural_data': cell_neural_data,
                
                # Additional useful columns
                'session': session,
                'plexon_session': plexon_session,
                'trial_number': trial_num,
                'trial_session': trial_row['trial_session']
            }
            
            unified_data.append(unified_row)
    
    unified_df = pd.DataFrame(unified_data)
    print(f"\nUnified DataFrame created with {len(unified_df)} rows")
    print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
    print(f"Unique trials: {unified_df['filename'].nunique()}")
    
    return unified_df

# Test with sample data first - using parallel version
sample_df = df.sample(10, random_state=42)
print("Testing parallel version with 10 sample trials...")
unified_df_corrected = create_unified_dataframe_parallel(
    sample_df, 
    cell_db, 
    chunk_size=5, 
    max_workers=2
)
unified_df_corrected

Testing parallel version with 10 sample trials...
Creating unified DataFrame with corrections (PARALLEL VERSION)...
Processing 10 trials and 5309 cells...
Split into 2 chunks of ~5 trials each
Using 2 parallel workers


Processing chunks: 100%|██████████| 2/2 [00:00<00:00, 33.90it/s]



Unified DataFrame created with 357 rows
Unique cells: 280
Unique trials: 10


,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,546,lfd,1,NaN,8,fi211003a.0316,STOP_R_SSD4,226.0,929,1157.0,...,1858,0.0,"[[232, 267], [453, 474], [1155, 1191]]",None,0,"[391.08, 1252.8, 1546.28, 1582.95, 1691.85, 17...",fi211003,a,316,fi211003a
1,547,hfdp,2,NaN,6,fi211003a.0316,STOP_R_SSD4,226.0,929,1157.0,...,1858,0.0,"[[232, 267], [453, 474], [1155, 1191]]",None,0,"[6.15, 12.53, 20.95, 28.15, 33.33, 39.55, 47.1...",fi211003,a,316,fi211003a
2,548,lfd,3,NaN,8,fi211003a.0316,STOP_R_SSD4,226.0,929,1157.0,...,1858,0.0,"[[232, 267], [453, 474], [1155, 1191]]",None,0,[],fi211003,a,316,fi211003a
3,549,hfdp,4,NaN,7,fi211003a.0316,STOP_R_SSD4,226.0,929,1157.0,...,1858,0.0,"[[232, 267], [453, 474], [1155, 1191]]",None,0,"[1.7, 9.1, 17.53, 27.08, 35.1, 41.73, 46.95, 5...",fi211003,a,316,fi211003a
4,550,hfdp,5,NaN,8,fi211003a.0316,STOP_R_SSD4,226.0,929,1157.0,...,1858,0.0,"[[232, 267], [453, 474], [1155, 1191]]",None,0,"[39.48, 59.28, 83.03, 124.15, 142.63, 176.23, ...",fi211003,a,316,fi211003a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,9782,msn,13,NaN,10,fi210819a.0957,STOP_R_SSD4,248.0,1052,1280.0,...,1981,0.0,"[[149, 185], [1300, 1334]]",None,0,"[1599.31, 1756.16]",fi210819,a,957,fi210819a
353,9784,msn,15,NaN,8,fi210819a.0957,STOP_R_SSD4,248.0,1052,1280.0,...,1981,0.0,"[[149, 185], [1300, 1334]]",None,0,"[110.16, 251.11, 468.61, 810.43, 1288.21, 1661...",fi210819,a,957,fi210819a
354,9785,msn,16,NaN,7,fi210819a.0957,STOP_R_SSD4,248.0,1052,1280.0,...,1981,0.0,"[[149, 185], [1300, 1334]]",None,0,"[1512.03, 1632.01]",fi210819,a,957,fi210819a
355,9787,msn,18,NaN,8,fi210819a.0957,STOP_R_SSD4,248.0,1052,1280.0,...,1981,0.0,"[[149, 185], [1300, 1334]]",None,0,"[379.01, 675.69, 716.06, 1177.66]",fi210819,a,957,fi210819a


In [17]:
unified_df_corrected['filename'].nunique(), sample_df['filename'].nunique()

(10, 10)

In [18]:
# sample_df

In [19]:
# Use the PARALLEL version for much faster processing
print("Using the PARALLEL unified DataFrame function...")
print("This function properly handles:")
print("1. MATLAB/Python indexing conversion")
print("2. Stability range filtering") 
print("3. Multiple stability ranges")
print("4. Parallel processing for speed")

# Run with full dataset - using optimal chunk size and max workers
print(f"\nProcessing full dataset with {len(df)} trials...")
unified_df = create_unified_dataframe_parallel(
    df, 
    cell_db, 
    chunk_size=200,  # Adjust based on memory vs speed tradeoff
    max_workers=None  # Use all available CPU cores
)

Using the PARALLEL unified DataFrame function...
This function properly handles:
1. MATLAB/Python indexing conversion
2. Stability range filtering
3. Multiple stability ranges
4. Parallel processing for speed

Processing full dataset with 110358 trials...
Creating unified DataFrame with corrections (PARALLEL VERSION)...
Processing 110358 trials and 5309 cells...
Split into 552 chunks of ~200 trials each
Using 20 parallel workers


Processing chunks: 100%|██████████| 552/552 [00:47<00:00, 11.71it/s]



Unified DataFrame created with 3185424 rows
Unique cells: 5252
Unique trials: 103077


In [20]:
# Analyze the unified DataFrame with additional columns
print("=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===")
print(f"Shape: {unified_df.shape}")
print(f"Memory usage: {unified_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nColumn info:")
print(f"Total columns: {len(unified_df.columns)}")
print(f"Columns: {list(unified_df.columns)}")

print(f"\nData summary:")
print(f"Unique cells: {unified_df['cell_ID'].nunique()}")
print(f"Unique trials: {unified_df['filename'].nunique()}")  
print(f"Unique sessions: {unified_df['session'].nunique()}")

print(f"\nCell types distribution:")
print(unified_df['cell_type'].value_counts())

print(f"\nTrial types in unified data:")
if 'type' in unified_df.columns:
    print(unified_df['type'].value_counts())

print(f"\nTrial name samples:")
trial_name_samples = unified_df['trial_name'].value_counts()
print(trial_name_samples.head(10))

print(f"\nNeural data statistics:")
neural_data_lengths = unified_df['neural_data'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
print(f"Mean spikes per trial: {neural_data_lengths.mean():.1f}")
print(f"Max spikes per trial: {neural_data_lengths.max()}")
print(f"Trials with no spikes: {(neural_data_lengths == 0).sum()}")

# Show sample rows with new columns
print(f"\nSample data with key columns:")
display_cols = ['cell_ID', 'cell_type', 'maestro_ID', 'filename', 'trial_name', 'type', 'reaction_time', 
                'first_relevant_saccade', 'trial_length', 'dir', 'problem']
available_cols = [col for col in display_cols if col in unified_df.columns]
print(f"Available columns: {available_cols}")
print(unified_df[available_cols].head(3))

# Check additional column data types and samples
print(f"\nAdditional column samples:")
additional_cols = ['segs_durations', 'segs_times', 'screen_rotation', 'saccades', 'blinks']
for col in additional_cols:
    if col in unified_df.columns:
        sample_val = unified_df[col].iloc[0]
        print(f"{col}: {type(sample_val).__name__} - {str(sample_val)[:100]}{'...' if len(str(sample_val)) > 100 else ''}")

=== UNIFIED DATAFRAME ANALYSIS (WITH ADDITIONAL COLUMNS) ===
Shape: (3185424, 27)
Memory usage: 4520.5 MB

Column info:
Total columns: 27
Columns: ['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename', 'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed', 'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade', 'segs_durations', 'segs_times', 'trial_length', 'screen_rotation', 'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session', 'trial_number', 'trial_session']

Data summary:
Unique cells: 5252
Unique trials: 103077
Unique sessions: 84

Cell types distribution:
cell_type
pu msn     1601672
msn         971238
hfdp        240455
gpi          93667
lfd          91789
pu tan       65631
tan          52246
unknown      17668
lfdb         14507
fsn          11264
fiber        10269
bd            5050
fiber?        4991
fef           4308
tan            543
ctx            126
Name: count, dtype: int64

Trial types in unified data:
type
GO  

In [21]:
print(unified_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

unified_df['grade'].value_counts()


cell_type
pu msn     2422
msn        1761
hfdp        466
lfd         155
gpi         140
tan          92
pu tan       87
lfdb         31
fsn          26
unknown      24
fiber        22
bd           10
fiber?        8
fef           6
tan           1
ctx           1
Name: count, dtype: int64


grade
8     1849206
9      686280
7      543571
6       83350
10      23017
Name: count, dtype: int64

In [22]:
# Get MSN cell IDs from cell_db
msn_cells_in_db = set(cell_db[cell_db['cell_type'] == 'msn']['cell_ID'].unique())

# Get MSN cell IDs from unified_df
msn_cells_in_unified = set(unified_df[unified_df['cell_type'] == 'msn']['cell_ID'].unique())

# Find MSN cells in cell_db but not in unified_df
msn_missing = msn_cells_in_db - msn_cells_in_unified

print(f"MSN cells in cell_db: {len(msn_cells_in_db)}")
print(f"MSN cells in unified_df: {len(msn_cells_in_unified)}")
print(f"MSN cells missing from unified_df: {len(msn_missing)}")
# the missing cells MSN cells are in non CSST trials or have no stable trials

MSN cells in cell_db: 1779
MSN cells in unified_df: 1761
MSN cells missing from unified_df: 18


In [23]:
unified_df[unified_df['cell_type'].isin(['msn']) & (unified_df['grade'] <= 8)]['cell_ID'].nunique()

1302

In [24]:
unified_df.attrs.update(df.attrs)
unified_df.attrs.update(cell_db.attrs)
unified_df.attrs

{'saccades_computation_params': {'speed_threshold': 50,
  'end_buffer': 0,
  'start_buffer': 0}}

In [ ]:
# Save the unified DataFrame
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
save_path.mkdir(exist_ok=True)

# Save as pickle for efficient loading
# pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
pickle_file = save_path / f'alt_saccade_params_{monkey}_cell_trial_data.pkl'
# unified_df.to_pickle(pickle_file)
print(f"Unified DataFrame saved to: {pickle_file}")

# # Also save a CSV version for easy inspection (but this will be larger)
# csv_file = save_path / f'unified_{monkey}_cell_trial_data.csv'
# # For CSV, convert neural_data to string representation to avoid issues
# csv_df = unified_df.copy()
# csv_df['neural_data'] = csv_df['neural_data'].apply(lambda x: str(x) if x else "[]")
# csv_df.to_csv(csv_file, index=False)
# print(f"CSV version saved to: {csv_file}")

# print(f"\nFile sizes:")
# print(f"Pickle: {pickle_file.stat().st_size / 1e6:.1f} MB") 
# print(f"CSV: {csv_file.stat().st_size / 1e6:.1f} MB")

# print(f"\nDataFrame ready for neural analysis!")
# print(f"Use: pd.read_pickle('{pickle_file}') to load the unified data")

Unified DataFrame saved to: /home/barak/Projects/population-analysis/data/unified_cell_trial_data/alt_saccade_params_fiona_cell_trial_data.pkl
